In [0]:
import csv
from pathlib import Path
import matplotlib.pyplot as plot

#NOTE: the notebook runs from its own folder (YOLO/src/analysis/graphs), so the logs are 3 folders up in YOLO/benchmarking
BENCH_DIR = Path.cwd().parent.parent.parent / "benchmarking"
GRAPH_DIR = BENCH_DIR / "graphs" #location to save graphs to
PLATFORM = "Jetson AGX Orin (CUDA)" #only used in graph titles

#adjust configuration of desired run
BATCH_SIZE = 1
STEP = None
OPTIMIZATION = None #e.g. "inductor"
PRECISION = None    #e.g. "bf16" or "fp32"

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
#read stage_timings.csv, one dict per row
#NOTE: rows are per image times in ms, columns:
#      timestamp,script,pass,device,batch_size,num_images,stage,substep,ms,step,optimization,precision

rows = list(csv.DictReader(open(BENCH_DIR / "stage_timings.csv")))
print(f"{len(rows)} rows in {BENCH_DIR / 'stage_timings.csv'}")
stages = {"pre_processing" : "#2a78d6", "inference" : "#eb6834", "post_processing" : "#1baf7a"} #dict to store each stage and their 
                                                                                                #associated color scheme

#helper func to find a run with desired configuration
def wanted_row(row):
    return ((BATCH_SIZE is None or row.get("batch_size") == str(BATCH_SIZE)) and (STEP is None or row.get("step") == str(STEP))
            and (OPTIMIZATION is None or row.get("optimization") == str(OPTIMIZATION)) and (PRECISION is None or row.get("precision") == PRECISION))

#helper func to find latest run
#NOTE: accepts pass_name as the type of harness run/pass associated with stored data point (e.g. substep_timing for substages,
#                                                                                           or wall_timing for stage wall times,
#                                                                                           or e2e for end to end batch times)
def latest_run(pass_name):
    matches = [r for r in rows if r["pass"] == pass_name and wanted_row(r)]
    if not matches:
        raise ValueError(f"NO {pass_name} ROWS MATCH THE SETTINGS (batch {BATCH_SIZE}, step {STEP}, opt {OPTIMIZATION}, precision {PRECISION})")
    latest = max(r["timestamp"] for r in rows) #finx most recent timestamp
    return [r for r in rows if r["timestamp"]==latest] #only grabs rows from most recent run

In [ ]:
#rows of most recent cuda events recorded in stage_timing.csv
run = [r for r in latest_run("substep_timing") if r["substep"] != "SUM_OF_STEPS"] #substeps only, in pipeline order
first = run[0] #grabs the first row taht matches cuda events
total = sum(float(r["ms"]) for r in run) #total time to find stage share percentages later on

for stage, color in stages.items():
    share = [r for r in run if r["stage"]==stage]
    bars = plot.barh([r["substep"] for r in share], [float(r["ms"]) for r in share], color = color, label = stage)
    plot.bar_label(bars, labels = [f"{float(r['ms']):.2f} ms ({float(r['ms']) / total * 100:.1f}%)" for r in share], padding = 3)


plot.xlim(0, max(float(r["ms"]) for r in run) * 1.35) #room for the labels past the longest bar
plot.xlabel("ms per image")
plot.title(f"{PLATFORM}: substep ASYNC latency per image (CUDA events)\n"
             f"batch {first['batch_size']}, step {first.get('step')}, {first.get('optimization')}, {first.get('precision')}, "
             f"{first['num_images']} images, {first['timestamp']}, total {total:.2f} ms/img")
plot.legend() #show what the colors are associated with from stages
plot.xlabel("ms per image") #add label
plot.gca().invert_yaxis() #make the earliest stages from preprocessing show up first top to bottom
plot.show()